In [ ]:
import torch
from transformers import AutoTokenizer
from arabert.preprocess import ArabertPreprocessor
# for text classification
from transformers import AutoModelForSequenceClassification

## araBERT Preprocessor

In [ ]:
model_name="bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name)

|## Initializing Model

In [ ]:
checkpoint = "aubmindlab/bert-base-arabert"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, return_dict=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

## Inference

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

In [ ]:
text = 'هذا مثال عربي اصطناعي لاختبار معالجة النصوص.'
probs = classify(text)
print(probs)

# Training Setup

## Dataset Prep

### AI

In [ ]:
with open('AI & H Reviews.csv', 'r', encoding='utf-16le') as file:
    content = file.read()

content_encoded = content.encode('windows-1256', errors='replace')

with open('path_to_converted_file.csv', 'wb') as file:
    file.write(content_encoded)


In [ ]:
import pandas as pd

# Load your CSV file
df = pd.read_csv('HR clean.csv')

# Add a column titled 'Label' with all values set to 1
df['Label'] = 1

# Save the modified DataFrame back to a CSV file
df.to_csv('modified_file1.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

# Assuming 'B.csv' is in the correct path and accessible
ai_df = pd.read_csv('modified_file.csv')
# Check the number of columns in ai_df and adjust the columns assignment accordingly
print("Number of columns in ai_df:", len(ai_df.columns))
ai_df.columns = ['text', 'label']  # Adjust this line if the number of columns is not 2

# Set all the labels to 0
ai_df['label'] = 0
ai_df = ai_df.drop(0)

# Load the 'Booking Hotel Reviews.csv' with correct path
human_df = pd.read_csv('modified_file1.csv')
human_df.columns = ['text', 'label']
# Set all the labels to 1
human_df['label'] = 1

# Merge the two dataframes
df = pd.concat([ai_df, human_df], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)
# Drop nan values
df = df.dropna()

# Save the dataframe to a csv file
df.to_csv('A.csv', index=False)


## Data Preprocessing

In [ ]:



from datasets import load_dataset

dataset = load_dataset("csv", data_files="AR_ART_with AI_E(windows-1256).csv", split="train" , encoding= 'windows-1256')

dataset = dataset.train_test_split(test_size=0.3)



In [ ]:
print(dataset)

In [ ]:
dataset["train"]

In [ ]:
dataset["train"][0]

### Applying araBERT Preprocessor

In [ ]:
def preprocess_function(examples):
    arabic_prep = arabert_prep.preprocess(examples["text"])
    result = tokenizer(arabic_prep,truncation=True,   
                       max_length=512, return_overflowing_tokens=True)

    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result
    # return tokenizer(arabic_prep, truncation=True, max_length=512, padding="max_length")

In [ ]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Setup Evaluation

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
def calculate_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    return {"accuracy": acc["accuracy"], "f1": f1, "recall": recall, "precision": precision}

## Training

In [ ]:
# find number of labels
num_labels = len(set(dataset["train"]["label"]))
print(num_labels)

# get label names
label_names = ["AI", "Human"]
print(label_names)

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

In [ ]:
print(id2label)
print(label2id)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id
)

In [ ]:
batch_size = 1
exp = "1"
training_args = TrainingArguments(
    output_dir="./trained/araBERT-base"+"_exp"+exp,
    learning_rate=2e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    save_total_limit=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    fp16=False,
    logging_dir="./logs/araBERT-base"+"_exp"+exp,
    logging_strategy = "epoch",
    logging_steps = 1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=calculate_metrics,
)

In [ ]:
 trainer.train()

## Final Inference

In [ ]:
from datasets import load_dataset
import torch

dataset = load_dataset("csv", data_files="old_data/ArabicTweets.csv", split="train")

dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
from transformers import AutoTokenizer
model_path = "trained/araBERT-base_exp1/checkpoint-206649"
tokenizer = AutoTokenizer.from_pretrained(model_path)

text = dataset["train"][0]["text"]

inputs = tokenizer(text, return_tensors="pt")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs.to(device)

with torch.no_grad():
    logits = model(**inputs).logits

In [ ]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

### Evaluation

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import precision_recall_fscore_support
import evaluate
import numpy as np
accuracy = evaluate.load("accuracy")
# import precision_recall_fscore_support
def evaluate_test_set(model, tokenizer, test_dataset, batch_size=32):
    if torch.cuda.is_available():
        device = "cuda"
    else :
        device = "cpu"
    print(f"Device: {device}")
    model.to(device)
    model.eval()
    acc = 0
    f1 = 0
    recall = 0
    precision = 0

    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)
    progress = tqdm(enumerate(test_dataloader), total=len(test_dataloader))
    for idx, batch in enumerate(test_dataloader):
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        scores = outputs.logits
        probs = scores.softmax(dim=1)
        predicted_class_ids = probs.argmax(dim=1).tolist()

        # batch_accuracy = sum([1 if label == prediction else 0 for label, prediction in zip(batch["label"], predicted_class_ids)])
        # batch_accuracy = batch_accuracy / len(batch["label"])

        batch_accuracy = accuracy.compute(predictions=predicted_class_ids, references=batch["label"])["accuracy"]
        batch_precision, batch_recall, batch_f1, _ = precision_recall_fscore_support(batch["label"], predicted_class_ids, average='weighted', labels=np.unique(predicted_class_ids))

        # print(batch_accuracy, batch_f1, batch_recall, batch_precision)
    
        acc += batch_accuracy
        f1 += batch_f1
        recall += batch_recall
        precision += batch_precision

        progress.update(1)
        progress.set_description(f"Batch {idx+1}/{len(test_dataloader)}: {batch_accuracy*100:.2f}%")
        # print(f"Batch {idx+1}/{len(test_dataloader)}: {batch_accuracy}")


    acc = acc / len(test_dataloader)
    f1 = f1 / len(test_dataloader)
    recall = recall / len(test_dataloader)
    precision = precision / len(test_dataloader)
    return acc, f1, recall, precision



In [ ]:
for example in dataset["test"]:
    print(example["text"])
    print(example["label"])
    break

In [ ]:
accuracy, f1, recall, precision =  evaluate_test_set(model, tokenizer, dataset["test"], batch_size=256)

In [ ]:
print(f"Accuracy: {accuracy*100:.2f}%, F1: {f1*100:.2f}%, Recall: {recall*100:.2f}%, Precision: {precision*100:.2f}%")

In [ ]:
import pandas as pd
from collections import Counter

# ضع مسار الملف هنا
file_path = 'مسار_ملفك_هنا.csv'

# قم بتحميل البيانات، استبدل 'windows-1256' بالترميز الصحيح لملفك إذا لزم الأمر
data = pd.read_csv(r"data/input.csv", encoding='windows-1256')


# دمج كل النصوص الموجودة في العمود 'text' إلى نص واحد كبير
all_text = ' '.join(data['text'])

# فصل النص إلى كلمات
words = all_text.split()

# حساب تكرار كل كلمة
word_counts = Counter(words)

# الحصول على الكلمة الأكثر تكرارًا وعدد التكرارات
most_common_word, count = word_counts.most_common(1)[0]

print(f"أكثر كلمة تكرارًا هي: '{most_common_word}' وتكررت {count} مرة/مرات.")


In [ ]:
import pandas as pd
from collections import Counter

# تعريف مسار الملف
file_path = r"data/input.csv" # استبدل بمسار الملف الفعلي

# قراءة الملف مع تحديد الترميز إذا لزم الأمر
# إذا واجهت مشكلة في الترميز يمكنك تجربة 'utf-8' أو 'windows-1256'
try:
    data = pd.read_csv(file_path, encoding='windows-1256')
except Exception as e:
    print(f"حدث خطأ أثناء قراءة الملف: {e}")
    exit()

# دمج كل النصوص الموجودة في عمود "text" إلى نص واحد كبير
all_texts = ' '.join(data['text'])

# تقسيم النص الكبير إلى كلمات باستخدام الفراغات
words = all_texts.split()

# حساب تكرار كل كلمة واسترجاع أكثر عشرين كلمة تكرارًا
most_common_words = Counter(words).most_common(20)

# طباعة أكثر عشرين كلمة تكرارًا
for word, frequency in most_common_words:
    print(f"الكلمة: '{word}', تكررت: {frequency} مرة/مرات")
